# Backpropagation and Reverse-Mode Automatic Differentiation
---
## Introduction

In this notebook, we study **backpropagation** through the lens of **reverse-mode automatic differentiation (autodiff)**. Rather than treating gradient computation as a black box, we build a minimal autodiff engine from scratch and use it to compute gradients over a small computational graph.

The goal is to understand, at a structural level, how modern deep learning frameworks (such as PyTorch or TensorFlow) compute gradients efficiently and exactly using the chain rule.

---

## Theoretical Background (Brief Overview)

### Computational Graphs

Any differentiable model can be represented as a **directed acyclic graph (DAG)** where:

- Nodes represent scalar values
- Edges represent operations (addition, multiplication, etc.)
- The final node represents a scalar output (e.g., loss)

Forward computation flows from inputs to output.

Backward computation (backpropagation) flows in the opposite direction.

---

### Chain Rule and Reverse-Mode Autodiff

Suppose a function is composed as:

$$
L = f(g(h(x)))
$$

By the chain rule:

$$
\frac{dL}{dx}
=
\frac{dL}{df}
\cdot
\frac{df}{dg}
\cdot
\frac{dg}{dh}
\cdot
\frac{dh}{dx}
$$

Backpropagation systematically applies this rule **in reverse order** across the computational graph.

Key ideas:

- Each node stores its local derivative rule.
- Gradients accumulate from multiple downstream paths.
- The backward pass requires reverse topological traversal of the graph.
- Reverse-mode autodiff is efficient when computing gradients of a scalar output with respect to many inputs — exactly the neural network setting.

---

## Purpose of the Example

In this notebook, we implement a minimal `Value` class that:

- Stores scalar data
- Stores gradient values
- Tracks parent nodes
- Stores a local backward function
- Performs reverse traversal using topological sorting

We then evaluate the expression:

```python
a = x * y
b = a + x
c = b * y
c.backward()

In [29]:
import math 

class Value:

    def __init__(self, data: int | float, _prev:set = ()):

        if not isinstance(data, (int, float)):
            raise TypeError("TypeError: data must be a int/float")
        
        self.grad = 0.0
        self.data = data
        self._prev = set(_prev)
        self._backward = lambda: None
    
    def __repr__(self):
        return f"Value(data={self.data}, grad={self.grad})"

    def __add__(self, other):

        if isinstance(other, (int, float)):
            other = Value(other)
        elif not isinstance(other, Value):
            raise TypeError("TypeError: addition operation only defined for Value and float types")

        output = Value(self.data + other.data, (self, other))
        
        def _backward():
            self.grad += output.grad
            other.grad += output.grad

        output._backward = _backward
        return output

    def __mul__(self, other):

        if isinstance(other, (int, float)):
            other = Value(other)
        elif not isinstance(other, Value):
            raise TypeError("TypeError: addition operation only defined for Value and float types")

        output = Value(self.data * other.data, (self, other))
        
        def _backward():
            self.grad += output.grad * other.data
            other.grad += output.grad * self.data

        output._backward = _backward
        return output
    
    def relu(self):
        output = Value(self.data if self.data > 0 else 0, (self,))
        
        def _backward():
            self.grad += output.grad * (1.0 if self.data > 0 else 0.0)

        output._backward = _backward
        return output

    def sigmoid(self):
        s = 1 / (1 + math.exp(-self.data))
        output = Value(s, (self,))

        def _backward():
            self.grad += output.grad * s * (1 - s)

        output._backward = _backward
        return output

    def backward(self):
        topo = []
        visited = set()

        def build(node):
            if node not in visited:
                visited.add(node)

                for parent in node._prev:
                    build(parent)
                
                topo.append(node)
        
        build(self)
        topo.reverse()
        self.grad = 1.0

        for node in topo:
            node._backward()

x = Value(2)
y = Value(3)

a = x * y
b = a + x
c = b * y
c.backward()

Given the computational graph:

$$
a = x y
$$
$$
b = a + x
$$
$$
c = b y
$$

We can rewrite the final expression explicitly as:

$$
c(x,y) = (xy + x)y
$$

Expanding:

$$
c(x,y) = xy^2 + xy
$$

---

### Partial Derivative with Respect to $x$

$$
\frac{\partial c}{\partial x}
=
\frac{\partial}{\partial x}(xy^2 + xy)
=
y^2 + y
$$

Evaluating at:

$$
x = 2, \quad y = 3
$$

$$
\frac{\partial c}{\partial x}
=
3^2 + 3
=
9 + 3
=
12
$$

---

### Partial Derivative with Respect to $y$

$$
\frac{\partial c}{\partial y}
=
\frac{\partial}{\partial y}(xy^2 + xy)
=
2xy + x
$$

Evaluating at:

$$
x = 2, \quad y = 3
$$

$$
\frac{\partial c}{\partial y}
=
2(2)(3) + 2
=
12 + 2
=
14
$$

---

## Final Expected Gradients

$$
\boxed{\partial_x c = 12.0}
$$

$$
\boxed{\partial_y c = 14.0}
$$

These analytical results must match the gradients computed by the reverse-mode autodiff engine:

```python
x.grad == 12.0
y.grad == 14.0

In [28]:
print('del_x:', x.grad, '\tdel_y:', y.grad)

del_x: 12.0 	del_y: 14.0


### MLP Example

In [31]:
# Inputs
x1 = Value(1.5)
x2 = Value(-2.0)

# Hidden Layer Parameters 
w1 = Value(0.8)
w2 = Value(-0.5)
b1 = Value(0.1)

w3 = Value(-1.0)
w4 = Value(1.2)
b2 = Value(-0.3)

# Hidden neuron 1
h1 = (x1 * w1 + x2 * w2 + b1).relu()

# Hidden neuron 2
h2 = (x1 * w3 + x2 * w4 + b2).relu()

# Output Layer Parameters 
w5 = Value(0.7)
w6 = Value(-1.1)
b3 = Value(0.05)

# Output neuron
y_hat = (h1 * w5 + h2 * w6 + b3).sigmoid()

# Backpropagation
y_hat.backward()

print("Output:", y_hat.data)
print("\nGradients:")
print("w1:", w1.grad)
print("w2:", w2.grad)
print("w3:", w3.grad)
print("w4:", w4.grad)
print("w5:", w5.grad)
print("w6:", w6.grad)
print("b1:", b1.grad)
print("b2:", b2.grad)
print("b3:", b3.grad)

Output: 0.8402380030563309

Gradients:
w1: 0.14095000634005217
w2: -0.18793334178673624
w3: 0.0
w4: 0.0
w5: 0.3087476329353524
w6: 0.0
b1: 0.09396667089336812
b2: 0.0
b3: 0.13423810127624017


## Gradient Flow Through ReLU: Active vs Inactive Neurons

During the 2-layer MLP experiment, we observed that several gradients were exactly zero:

- `w3`, `w4`, `b2` (weights of hidden neuron 2)
- `w6` (its outgoing weight)

This behavior was not caused by vanishing gradients. Instead, it resulted from the **ReLU activation function gating the gradient**.

---

### What Happened?

For the second hidden neuron:

$$
z_2 = x_1 w_3 + x_2 w_4 + b_2
$$

With the chosen parameters, we obtained:

$$
z_2 < 0
$$

Since ReLU is defined as:

$$
\text{ReLU}(z) = \max(0, z)
$$

we had:

$$
h_2 = \text{ReLU}(z_2) = 0
$$

The derivative of ReLU is:

$$
\frac{d}{dz}\text{ReLU}(z) =
\begin{cases}
1 & z > 0 \\
0 & z \le 0
\end{cases}
$$

Because \( z_2 < 0 \), the derivative was:

$$
\text{ReLU}'(z_2) = 0
$$

This completely blocked gradient flow through that neuron.

---

### Consequence for the Gradients

If the local derivative is zero:

$$
\frac{\partial L}{\partial z_2} = 0
$$

Then, by the chain rule:

- All incoming weights to that neuron receive zero gradient.
- Any outgoing weights connected to that neuron also receive zero gradient (since the activation is zero).

This explains why:

$$
w_3 = w_4 = b_2 = w_6 = 0
$$

---

### Important Distinction: Not Vanishing Gradients

This phenomenon is **not** the classical vanishing gradient problem.

- Vanishing gradients occur when derivatives are small and shrink progressively across deep layers (e.g., sigmoid saturation).
- In this case, the gradient was exactly zero due to the ReLU derivative being zero.

This is known as the **dead neuron problem** in ReLU networks.

---

### Key Insight

ReLU does not gradually shrink gradients like sigmoid; instead, it either:

- Passes gradients unchanged (if active),
- Or blocks them completely (if inactive).

This experiment illustrates how nonlinear activations directly control gradient flow in neural networks.